In [ ]:
from utils import *
import wandb
from datetime import datetime

In [ ]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [ ]:
def saveExperiment(imageModel, textModel, config, experimentName, start):
    latestPath = os.path.join("checkpoints", "finetune", "latest")
    if not os.path.exists(os.path.join("checkpoints", "finetune", "latest")):
        os.mkdir(latestPath)

    stamp = start.strftime("%Y-%m-%d %H-%M")
    timePath = os.path.join("checkpoints", "finetune", stamp)
    if not os.path.exists(timePath):
        os.mkdir(timePath)

    saveToPath(latestPath, imageModel, textModel, config, experimentName)
    saveToPath(timePath, imageModel, textModel, config, experimentName)


def saveToPath(path, imageModel, textModel, config, experimentName):
    if not os.path.exists(os.path.join(path, experimentName)):
        os.mkdir(os.path.join(path, experimentName))

    torch.save(imageModel.state_dict(), os.path.join(path, experimentName, "image.pt"))
    # textModel.save_pretrained(os.path.join(path, experimentName, "text"))
    torch.save(textModel.state_dict(), os.path.join(path, experimentName, "text.pt"))
    config.save(os.path.join(path, experimentName, "config.json"))

In [ ]:
def getTrainableParams(model):
    return [p for p in model.parameters() if p.requires_grad]

def setFrozen(model, frozen: bool):
    for param in model.parameters():
        param.requires_grad = not frozen

def unfreezeTopBlocks(imageModel, textModel: CLIPTextEmbedder, n: int):
    """
    Unfreeze the last n transformer blocks of both models and keep both heads trainable.
    Image model: self.transformer.layers
    Text model:  self.transformer.layers
    """

    def freezeLayers(layers):
        for layer in layers[-n:]:
            for param in layer.parameters():
                param.requires_grad = True

    setFrozen(imageModel, True)
    layers = imageModel.model.transformer.layers
    freezeLayers(layers)

    for param in imageModel.head.parameters():
        param.requires_grad = True

    setFrozen(textModel, True)
    layers = textModel.model.text_model.encoder.layers
    freezeLayers(layers)

    for param in textModel.head.parameters():
        param.requires_grad = True

def getParamGroups(model, headLR, layerLR):
    """
    Split a model's trainable params into head vs transformer layer groups.
    """
    headParamIds = {id(p) for p in model.head.parameters()}

    headParams = [
        p for p in model.parameters()
        if p.requires_grad and id(p) in headParamIds
    ]
    layerParams = [
        p for p in model.parameters()
        if p.requires_grad and id(p) not in headParamIds
    ]

    return [
        {"params": headParams,  "lr": headLR},
        {"params": layerParams, "lr": layerLR},
    ]

def makeOptimizer(config, imageModel, textModel):
    layerLR = 0.01 * config.learningRate

    paramGroups = (
        getParamGroups(imageModel, config.learningRate, layerLR)
        + getParamGroups(textModel, config.learningRate, layerLR)
    )

    return torch.optim.Adam(paramGroups)

In [ ]:
def trainModel(config, textModel, imageModel, dataset, experimentName, start, imageConfig):
    setFrozen(imageModel, True)
    setFrozen(textModel, True)
    for param in imageModel.head.parameters():
        param.requires_grad = True
    for param in textModel.head.parameters():
        param.requires_grad = True
    optimizer = makeOptimizer(config, imageModel, textModel)

    imageModel.to(device)
    textModel.to(device)

    queue = MoCoQueue(dim=config.sharedDim, size=config.queueSize)
    objective = EmbeddingLoss(temperature=0.04, alpha=0.1)
    criterion = Perplexity(objective)

    train, test = dataset.split(dataset, batchSize=config.batchSize)

    testIter = itertoolsBetter(test)

    testHistory = []

    total = 0

    run = None

    epochPhases = [
        (config.headOnlyEpochs, None),
        (config.phase1Epochs, 1),
        (config.phase2Epochs, 2),
        (config.fullEpochs, len(imageModel.model.transformer.layers)),
    ]

    try:
        for phaseEpochs, nBlocks in epochPhases:
            if phaseEpochs == 0:
                continue
            if nBlocks is None:
                setFrozen(imageModel, True)
                setFrozen(textModel, True)
                for param in imageModel.head.parameters():
                    param.requires_grad = True
                for param in textModel.head.parameters():
                    param.requires_grad = True
            else:
                unfreezeTopBlocks(imageModel, textModel, nBlocks)
            optimizer = makeOptimizer(config, imageModel, textModel)

            for epoch in range(phaseEpochs):
                progress = 0
                averageTrainLoss = 0
                averageTestLoss = 0
                for images, info, text, _ in train:
                    imageModel.train()
                    textModel.train()
                    optimizer.zero_grad()

                    imageOutputs = imageModel(images.to(device))
                    textOutputs = textModel(text.to(device))
                    loss = objective(imageOutputs, textOutputs, info.to(device), queue=queue)
                    trainPerplexity = criterion(imageOutputs, textOutputs, info.to(device), queue=queue)

                    trainLoss = loss["total"].detach().item()
                    averageTrainLoss = (averageTrainLoss * progress + trainLoss) / (progress + 1)

                    loss["total"].backward()
                    optimizer.step()

                    with torch.no_grad():
                        imageModel.eval()
                        textModel.eval()
                        images1, info1, text1, _ = next(testIter)
                        imageOutputs1 = imageModel(images1.to(device))
                        textOutputs1 = textModel(text1.to(device))
                        loss1 = objective(imageOutputs1, textOutputs1, info1.to(device), queue=queue)
                        testPerplexity = criterion(imageOutputs1, textOutputs1, info1.to(device), queue=queue)

                        testLoss = loss1["total"].detach().item()
                        averageTestLoss = (averageTestLoss * progress + testLoss) / (progress + 1)

                    if run is None:
                        run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                    
                    run.log({"Train Loss": trainLoss, 
                             "Train Perplexity": trainPerplexity.detach().item(),
                             "Train InfoNCE": loss["info"].detach().item(),
                             "Train SigREG": loss["sig"].detach().item(),
                             "Test Loss": testLoss,
                             "Test Perplexity": testPerplexity.detach().item(),
                             "Test InfoNCE": loss1["info"].detach().item(),
                             "Test SigREG": loss1["sig"].detach().item(),
                             "Train Recall@1": recallAtK(imageOutputs, textOutputs, k=1, families=info.to(device), queue=queue),
                             "Test Recall@1": recallAtK(imageOutputs1, textOutputs1, k=1, families=info1.to(device), queue=queue),
                             "Train Recall@5": recallAtK(imageOutputs, textOutputs, k=5, families=info.to(device), queue=queue),
                             "Test Recall@5": recallAtK(imageOutputs1, textOutputs1, k=5, families=info1.to(device), queue=queue),
                             "Train Recall@10": recallAtK(imageOutputs, textOutputs, k=10, families=info.to(device), queue=queue),
                             "Test Recall@10": recallAtK(imageOutputs1, textOutputs1, k=10, families=info1.to(device), queue=queue)
                             }, step=total)

                    progress += 1
                    total += 1

                    progressString = f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}%"

                    print(f"{progressString} |  Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}",end="")

                print(f"\rEpoch {epoch + 1} | Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}{' ' * 50}")

                testHistory.append(averageTestLoss)

    except KeyboardInterrupt:
        saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
        return imageModel, textModel

    saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
    return imageModel, textModel

In [ ]:
queryConfig = Config().load(os.path.join("configs", "querying.json"))

In [ ]:
sharedDim = queryConfig.sharedDim

textModelName = "openai/clip-vit-base-patch32"
textModel = CLIPTextEmbedder(textModelName, sharedDim).to(device)

# textModelName = "bert-base-uncased"
# textModel = BERTTextEmbedder(textModelName, sharedDim).to(device)

vit, imageConfig = ViT.load(os.path.join("checkpoints", "pretrain", "2026-06-03 04-37"))
imageModel = ViTEmbedder(vit, sharedDim).to(device)

dataset = CombinedQueryData(imageConfig.dataset, training=True)
dataset.method = "none"

dataset.setTokenizer(textModelName)

experimentName = "ViT" + " " + textModelName.replace("/", "-")

imageModel, textModel = trainModel(queryConfig, textModel, imageModel, dataset, experimentName, datetime.now(), imageConfig)

saveExperiment(imageModel, textModel, imageConfig, experimentName, datetime.now())


Loading MyFonts images from dataset ====================


Fonts serialized: 1855/3794google\fonts\ofl\kumarone\KumarOne-Regular.ttf execution context too long
Fonts serialized: 2335/3794google\fonts\ofl\notocoloremojicompattest\NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3735/3794google\fonts\ofl\zcoolxiaowei\ZCOOLXiaoWei-Regular.ttf [Errno 22] Invalid argument: 'google\\bitmaps\\????? ?? al.bmp'
Fonts serialized: 3794/3794


Fonts serialized: 36/18623dafont\fonts\135atom_sans.ttf [Errno 2] No such file or directory: 'dafont\\bitmaps\\13/5Atom Sans Regular al.bmp'
Fonts serialized: 79/18623dafont\fonts\50fifty.ttf [Errno 2] No such file or directory: 'dafont\\bitmaps\\50/fifty Regular al.bmp'
Fonts serialized: 353/18623dafont\fonts\adrenaline_zero.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Adrenaline : Zero Adrenaline : Zero al.bmp'
Fonts serialized: 410/18623dafont\fonts\aggstock.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\?ggstock Regula

cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.
cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.


Fonts serialized: 3910/18623dafont\fonts\doilie.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\do i lie? liars al.bmp'
Fonts serialized: 3998/18623dafont\fonts\dragonforce.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\?DragonForcE? Regular al.bmp'
Fonts serialized: 4001/18623dafont\fonts\dragos.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Dragos Brush ? al.bmp'
Fonts serialized: 4307/18623dafont\fonts\elegantech-.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\EleganTech? Regular al.bmp'
Fonts serialized: 4346/18623dafont\fonts\elsö.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Els? Regular al.bmp'
Fonts serialized: 4872/18623

cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.
cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.


Fonts serialized: 5833/18623dafont\fonts\gomarice_whats_love_oshare.ttf [Errno 22] Invalid argument: "dafont\\bitmaps\\What's Love? Oshare Regular al.bmp"
Fonts serialized: 6011/18623dafont\fonts\groovenext-.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\GrooveNext? Regular al.bmp'
Fonts serialized: 6553/18623dafont\fonts\humbug.ttf corrupt cmap table format 0 (data length: 534, header length: 598)
Fonts serialized: 6568/18623

3 extra bytes in post.stringData array


Fonts serialized: 7066/18623dafont\fonts\jerónimo_cartoon.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Jer?nimo cartoon Regular al.bmp'
Fonts serialized: 7162/18623dafont\fonts\julia_black_extended.ttf Not a TrueType or OpenType font (bad sfntVersion)
Fonts serialized: 7317/18623dafont\fonts\kaylafiz.ttf division by zero
Fonts serialized: 7979/18623dafont\fonts\lavoshandy_99.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\LavosHandy? Regular al.bmp'
Fonts serialized: 8089/18623dafont\fonts\lettif__.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Letters II "Fenotype" al.bmp'
Fonts serialized: 8161/18623dafont\fonts\like_giselle.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Like Giselle? Regular al.bmp'
Fonts serialized: 8285/18623dafont\fonts\lomonesia-_dingbats.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Lomonesia? Dingbats Regular al.bmp'
Fonts serialized: 8322/18623dafont\fonts\louis_george_cafe.ttf [Errno 22] Invalid argument: 'dafont\\bitmaps\\Louis George 

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\dylan\_netrc.
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1 | Train Loss: 4.79 | Test Loss: 4.80                                                  
Epoch 2 | Train Loss: 4.29 | Test Loss: 4.58                                                  
Epoch 1 | Train Loss: 3.83 | Test Loss: 4.49                                                  
Epoch 2 | Train Loss: 3.50 | Test Loss: 4.53                                                  
Epoch 1 | Train Loss: 3.28 | Test Loss: 4.64                                                  
Epoch 2 | Train Loss: 3.06 | Test Loss: 4.80                                                  
Epoch 1 | Train Loss: 2.95 | Test Loss: 4.85                                                  
Epoch 2 | Train Loss: 2.69 | Test Loss: 5.03                                                  
Epoch 3 | Train Loss: 2.48 | Test Loss: 5.21                                                  
Epoch 4 | Train Loss: 2.33 | Test Loss: 5.31                                                  
